## Setup — Import Libraries
All topologicpy modules needed for geometry, topology, graphs, and dictionaries.

In [ ]:
# Import the needed libraries
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## Version Check
Confirm topologicpy meets the minimum required version.

In [ ]:
# Check the TopologicPy Version
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## Renderer Configuration
Set render target. Options: `vscode` · `colab` · `browser`.

In [ ]:
# Set my renderer: ( Visual studio code: "vscode" /  Google Colab: "colab" / Browser: "browser" )
renderer = "vscode"

## 01 — Load Room Geometry
Import the extruded room volumes from Rhino as a list of topology objects.

In [ ]:
# Import the OBJ file as a Topology object
objects = Topology.ByOBJPath(r"C:\Users\chidi\OneDrive\Desktop\macad\module-03\aia-gml\assets\ROOMS-GEO.obj")
print(objects)

## 02 — Classify Rooms & Assign Spatial Categories
Each room is classified by name keyword into one of four categories:

| Category | Color | Spaces |
|---|---|---|
| `public` | green gradient | lobby, entry |
| `circulation` | red | corridors, stairs |
| `cell` | black | detainee spaces |
| `private` | yellow gradient | offices, staff |

> **To resize room nodes:** change `SIZE_ROOM` in this cell.

In [ ]:
cells = []
selectors = []
room_data = []

for object in objects:
    d = Topology.Dictionary(object)
    faces = Topology.Faces(object)
    if len(faces) > 1:
        c = Cell.ByFaces(faces)
        c = Topology.RemoveCollinearEdges(c)
        s = Topology.InternalVertex(c)
        name = Dictionary.ValueAtKey(d, "name") or ""
        name_lower = name.lower()
        if any(k in name_lower for k in ["lobby", "public", "security"]):
            category = "public"
        elif any(k in name_lower for k in ["vertical", "circulation"]):
            category = "circulation"
        elif "cell" in name_lower:                                         
            category = "cell"                                               
        else:
            category = "private"
        room_data.append((c, s, d, category))

counts = {"public": 0, "circulation": 0, "cell": 0, "private": 0}        
for _, _, _, cat in room_data:
    counts[cat] += 1
indices = {"public": 0, "circulation": 0, "cell": 0, "private": 0}       

def lerp_color(c1, c2, t):
    return "#{:02x}{:02x}{:02x}".format(
        int(c1[0] + (c2[0] - c1[0]) * t),
        int(c1[1] + (c2[1] - c1[1]) * t),
        int(c1[2] + (c2[2] - c1[2]) * t)
    )

green_light  = (180, 240, 180)
green_dark   = (0,   100,   0)
yellow_light = (255, 255, 153)
yellow_dark  = (204, 140,   0)

for c, s, d, category in room_data:
    idx = indices[category]
    count = counts[category]
    t = idx / max(count - 1, 1)

    if category == "public":
        color = lerp_color(green_light, green_dark, t)
    elif category == "circulation":
        color = "red"
    elif category == "cell":                                         
        color = "black"                       
    else:
        color = lerp_color(yellow_light, yellow_dark, t)

    indices[category] += 1
    SIZE_ROOM = 35  # ← change this number to resize all room nodes
    d = Dictionary.SetValuesAtKeys(d, ["color", "vertex_size"], [color, SIZE_ROOM])
    s = Topology.SetDictionary(s, d)
    selectors.append(s)
    cells.append(c)
    print(Dictionary.Keys(d), Dictionary.Values(d))

print(len(cells))

## 03 — Build CellComplex & Transfer Dictionaries
Merge all room cells into one CellComplex and push color/size dictionaries from selectors onto each cell.

In [ ]:
house = CellComplex.ByCells(cells) # create a cell complex from the cells
house = Topology.TransferDictionariesBySelectors(house, selectors, tranCells=True) # transfer the dictionaries from the selectors to the cells
house_cells = Topology.Cells(house) # get the cells of the cell complex to check if the dictionaries are transferred correctly, and to print their keys and values
for house_cell in house_cells:
    d = Topology.Dictionary(house_cell)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 04 — Visualize Building Geometry
Render 3D volumes colored by spatial category. White edges separate volumes cleanly.

In [ ]:
# Show the house
Topology.Show(house_cells, 
              faceColorKey="color", 
              faceOpacity=1.0,
              opacityKey="nothing",
              faceOpacityKey="nothing",
              backgroundColor="white",  
              width=700,
              height=500,
              edgeColor="white")

## 05 — Primal Graph: Geometric Adjacency
**Nodes** = rooms · **Edges** = any shared face (wall-to-wall contact), regardless of doors.

Node size encodes **degree** (number of shared faces): spatially central rooms appear larger. This graph reveals geometric proximity — not access.

In [ ]:
# Primal graph: nodes = rooms, edges = shared face (geometric proximity, no door required)
g = Graph.ByTopology(house, useInternalVertex=True)  # True ensures vertex stays inside the volume

# First pass: collect all degrees to normalize sizes across the range
verts = Graph.Vertices(g)
degrees = [Graph.VertexDegree(g, v) for v in verts]  # count edges per vertex
min_deg = min(degrees)                                 # lowest connectivity in the building
max_deg = max(degrees)                                 # highest connectivity in the building

# Second pass: assign normalized size and print
for v, degree in zip(verts, degrees):
    d = Topology.Dictionary(v)                         # get the vertex dictionary
    # map degree to size range 10–30 so difference is visible but not overwhelming
    size = 10 + (degree - min_deg) / max(max_deg - min_deg, 1) * 20
    d = Dictionary.SetValuesAtKeys(d, ["vertex_size"], [size])  # update size in dictionary
    v = Topology.SetDictionary(v, d)                   # write dictionary back to vertex
    print(Dictionary.Keys(d), Dictionary.Values(d))    # confirm keys and values

## 06 — Visualize Primal Graph
Primal graph overlaid on transparent building. Grey edges = raw geometric adjacency, no access logic applied.

In [ ]:
# Show primal graph overlaid on the building geometry
Topology.Show(g, house,
              vertexSizeKey="vertex_size",   # drives node size from the degree-based value above
              vertexColorKey="color",         # drives node color from spatial category (public/private/etc)
              edgeColor="grey",               # all edges red — encodes geometric adjacency uniformly
              faceOpacity=0.15,              # near-transparent geometry so graph reads clearly
              backgroundColor="white",
              width=800,
              height=500)

## 07 — Load Apertures
Three aperture types loaded as separate lists so they can be combined selectively per graph.

| Type | Color | Role |
|---|---|---|
| Solid door | burgundy | movement only |
| Glass door | cyan | movement + visual |
| Window | dark blue | visual only |

> **To resize aperture nodes:** change `SIZE_DOOR`, `SIZE_GLASS_DOOR`, `SIZE_WINDOW` at the top of this cell.

In [ ]:
# ─── APERTURE NODE SIZES ───────────────────────────────────────────────────
# Change these three numbers to resize aperture nodes across both graphs.
# Keep them smaller than SIZE_ROOM (35) so rooms read as the dominant nodes.
SIZE_DOOR       = 10  # solid doors   → burgundy
SIZE_GLASS_DOOR = 10  # glass doors   → cyan
SIZE_WINDOW     = 10  # windows       → dark blue
# ────────────────────────────────────────────────────────────────────────────

# Load each aperture type into separate lists for graph splitting later
doors = []
glass_doors = []
windows = []

# Solid doors — physical passage only
objects = Topology.ByOBJPath(r"C:\Users\chidi\OneDrive\Desktop\macad\module-03\aia-gml\assets\DOORS-GEO.obj")
faces = [Topology.Faces(object)[0] for object in objects]
for face in faces:
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["type", "color", "vertex_size"], ["door", "#800020", SIZE_DOOR])    # burgundy
    face = Topology.SetDictionary(face, d)
    doors.append(face)

# Glass doors — passage AND visual connection
objects = Topology.ByOBJPath(r"C:\Users\chidi\OneDrive\Desktop\macad\module-03\aia-gml\assets\GLASS-DOORS-GEO.obj")
faces = [Topology.Faces(object)[0] for object in objects]
for face in faces:
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["type", "color", "vertex_size"], ["glass_door", "cyan", SIZE_GLASS_DOOR]) # cyan
    face = Topology.SetDictionary(face, d)
    glass_doors.append(face)

# Windows — visual connection only
objects = Topology.ByOBJPath(r"C:\Users\chidi\OneDrive\Desktop\macad\module-03\aia-gml\assets\WINDOWS-GEO.obj")
faces = [Topology.Faces(object)[0] for object in objects]
for face in faces:
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["type", "color", "vertex_size"], ["window", "darkblue", SIZE_WINDOW])    # dark blue
    face = Topology.SetDictionary(face, d)
    windows.append(face)

apertures = doors + glass_doors + windows  # combined list kept for reference
print(len(doors), "doors |", len(glass_doors), "glass doors |", len(windows), "windows")

## 08 — Build Movement & Visual House Models
Two versions of the house, each loaded with only the relevant apertures:
- `house_movement` → doors + glass doors (physical passage)
- `house_visual` → glass doors + windows (line of sight)

In [ ]:
# Two separate houses — each carrying only the apertures relevant to that graph
house_movement = Topology.AddApertures(house, doors + glass_doors, subTopologyType="face")  # physical access only
house_visual   = Topology.AddApertures(house, glass_doors + windows, subTopologyType="face") # visual connection only

## 09 — Movement Graph: Physical Access
Dual graph built from `house_movement`. An edge exists **only where a door or glass door** connects two rooms.

`direct=False` ensures rooms sharing only a wall get no edge — security boundaries are topologically enforced.

In [ ]:
# Movement graph: edges form only where a solid door or glass door exists between rooms
g_movement = Graph.ByTopology(house_movement, direct=False, viaSharedApertures=True, toExteriorApertures=False)

# Print vertex data for both graphs to verify color and size are correct
print("--- Movement Graph ---")
for v in Graph.Vertices(g_movement):
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 10 — Visualize Movement Graph
Rooms colored by zone. Aperture nodes (burgundy = door · cyan = glass door) sit on walls, showing where physical movement is possible.

In [ ]:
# Movement graph: who can physically reach whom — rooms (colored by zone) + door/glass-door aperture nodes
Topology.Show(house_movement, doors + glass_doors, g_movement,
              vertexSizeKey="vertex_size",   # rooms=20, apertures=10
              vertexColorKey="color",         # rooms by zone, doors=burgundy, glass doors=cyan
              edgeColor="gray",              # neutral edges, aperture node color carries the meaning
              faceOpacity=0.15,
              backgroundColor="white",
              width=700,
              height=500)

## 11 — Visual Graph: Line of Sight
Dual graph built from `house_visual`. Edges form where **glass doors or windows** exist.

`toExteriorApertures=True` connects rooms to the exterior through windows — revealing surveillance reach.

In [ ]:
# Visual graph: edges form where glass doors or windows exist — includes exterior connections via windows
g_visual = Graph.ByTopology(house_visual, direct=False, viaSharedApertures=True, toExteriorApertures=True)

print("--- Visual Graph ---")
for v in Graph.Vertices(g_visual):
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 12 — Visualize Visual Graph
Rooms colored by zone. Aperture nodes (cyan = glass door · blue = window) mark visual thresholds. Dense connections signal high visual control.

In [ ]:
# Visual graph: who can see whom — rooms + glass-door/window aperture nodes
Topology.Show(house_visual, glass_doors + windows, g_visual,
              vertexSizeKey="vertex_size",   # rooms=20, apertures=10
              vertexColorKey="color",         # rooms by zone, glass doors=cyan, windows=blue
              edgeColor="gray",
              faceOpacity=0.15,
              backgroundColor="white",
              width=700,
              height=500)